In [1]:
print(123)

123


In [16]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [17]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [18]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [20]:
answer = assistant.rag('how do i run Ollama locally?')
print (answer)

To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download  
   - Choose your OS:
     - **macOS**: download the `.pkg` and install it
     - **Windows**: download the `.msi` and install it
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Run a model**
   - Open a terminal and run:
     ```bash
     ollama run llama3
     ```
   This downloads LLaMA 3 (about 4GB), starts it locally, and opens a chat interface.

3. **(Optional) Test the local Ollama server**
   ```bash
   curl http://localhost:11434
   ```
   You should get a JSON response like:
   ```json
   {"models": [...]}
   ```

4. **(Optional) Use it from Python**
   - Install the client:
     ```bash
     pip install ollama
     ```
   - Minimal example:
     ```python
     import ollama

     response = ollama.chat(
         model='llama3',
         messages=[{"role": "user", "content": your_prompt}]
     )

     print(response['message'][

In [21]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open and you meet any prerequisites, you can often join after discovering it.\n\nA few quick things to check:\n- **Enrollment deadline:** Is registration still open?\n- **Prerequisites:** Do you need prior experience or approval?\n- **Capacity:** Is the class full?\n- **Start date/access:** Have lectures or materials already begun?\n\nIf you want, I can help you draft a short message to the instructor or course admin asking to join.'

In [22]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [23]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"just discovered the course Can I join it"}', call_id='call_aS8B9TLIx20Fsff24mdBYhUT', name='search', type='function_call', id='fc_0659eeefec597652006a4c9c7fe358819bba148c8d653dffa5', namespace=None, status='completed')]


In [26]:
call = response.output[0]

In [27]:
call

ResponseFunctionToolCall(arguments='{"query":"just discovered the course Can I join it"}', call_id='call_aS8B9TLIx20Fsff24mdBYhUT', name='search', type='function_call', id='fc_0659eeefec597652006a4c9c7fe358819bba148c8d653dffa5', namespace=None, status='completed')

In [28]:
import json

args= json.loads(call.arguments)
args

{'query': 'just discovered the course Can I join it'}

In [29]:
call.name

'search'

In [30]:
results = search(**args)

In [31]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that decide which tools to call. The goal is to se

In [32]:
result_json = json.dumps(results, indent=2)

In [33]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "193612db63",
    "course": "llm-zoomcamp",
    "section": "Module 3: Orchestration",
    "question": "Why do we need orchestration / Kestra \u2014 can't I just run the code in a notebook?",
    "answer": "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that de

In [34]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [36]:
messages.append (call)

In [ ]:
#send entire history to LLM, append to first call
messages.append(function_call_output)

In [39]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"just discovered the course Can I join it"}', call_id='call_aS8B9TLIx20Fsff24mdBYhUT', name='search', type='function_call', id='fc_0659eeefec597652006a4c9c7fe358819bba148c8d653dffa5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_aS8B9TLIx20Fsff24mdBYhUT',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "193612db63",\n    "course": "llm-zoomcamp",\n    "section": "Module 3: Orchestration",\n    "question": "Why do we need orchestration / Kestra \\u2014 can\'t I just run the code in a notebook?",\n    "answer": "No

In [41]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes—you can still join. If you want to receive a certificate, you’ll need to submit your project while the course is still accepting submissions.'

In [42]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(972, 33)

In [50]:
#converting the json string to python dictionary
#helper function used in an LLM agent workflow to execute a tool requested by the AI model and return the result back to it

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [51]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [52]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=messages,
    tools=[search_tool],
)

User
 |
 v
LLM
 |
 | decides:
 | "I need search()"
 |
 v
make_call()
 |
 | runs
 |
 v
search(query="...")
 |
 v
returns results
 |
 v
LLM
 |
 v
Final answer

ResponseFunctionToolCall(arguments='{"query":"join the course discovered the course can I join it late enrollment"}', call_id='call_RAHSXHwMrarOIOIrBfGysHj2', name='search', type='function_call', id='fc_01cc05ff021d3290006a4ca71c2284819ba6d424f3bd9ed86d', namespace=None, status='completed')

In [62]:
messages.extend(response.output)

for item in response.output:
    print(item)

    if item.type == 'function_call':
        #process the function call
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        
    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ResponseFunctionToolCall(arguments='{"query":"join the course discovered the course can I join it late enrollment"}', call_id='call_RAHSXHwMrarOIOIrBfGysHj2', name='search', type='function_call', id='fc_01cc05ff021d3290006a4ca71c2284819ba6d424f3bd9ed86d', namespace=None, status='completed')
function_call: search {"query":"join the course discovered the course can I join it late enrollment"}
ResponseFunctionToolCall(arguments='{"query":"course enrollment after start can I join"}', call_id='call_Z6qLU7kP1LoySh4I22YF8iJ0', name='search', type='function_call', id='fc_01cc05ff021d3290006a4ca71c2290819bab794fb236e73087', namespace=None, status='completed')
function_call: search {"query":"course enrollment after start can I join"}
ResponseFunctionToolCall(arguments='{"query":"registration deadline add student late start"}', call_id='call_BTjCCq3EXSwKJsRykWWNFspQ', name='search', type='function_call', id='fc_01cc05ff021d3290006a4ca71c2298819b9ce73f2477354738', namespace=None, status='completed

In [63]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered the course can I join it late enrollment"}', call_id='call_RAHSXHwMrarOIOIrBfGysHj2', name='search', type='function_call', id='fc_01cc05ff021d3290006a4ca71c2284819ba6d424f3bd9ed86d', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment after start can I join"}', call_id='call_Z6qLU7kP1LoyS

In [64]:

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course enroll registration discovered course can I join"}
function_call: search {"query":"course FAQ enrollment join course discovered late can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

A couple of notes:
- You can start learning right away.
- If you want a certificate, you need to submit your project while submissions are still open.
- Certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help you figure out the best way to start the course now. Any other areas you want to explore?


In [68]:
def agent_loop(instructions, question, model="gpt-5.4-nano") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="gpt-5.4-nano",
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [69]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [70]:
agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join the course can I join it discovered the course late enrollment"}
iteration #2...
ASSISTANT:
Yes—you can still join. Since you just discovered the course, you should be able to start now, and if you want a **certificate**, you’ll need to **submit your project while we’re still accepting submissions**.

If you tell me whether you care about the certificate (or just want to learn), I can point you to the best next steps. Are there other areas you want to explore (schedule, deadlines, or how to submit)?


'Yes—you can still join. Since you just discovered the course, you should be able to start now, and if you want a **certificate**, you’ll need to **submit your project while we’re still accepting submissions**.\n\nIf you tell me whether you care about the certificate (or just want to learn), I can point you to the best next steps. Are there other areas you want to explore (schedule, deadlines, or how to submit)?'

In [74]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.


At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I couldn’t find anything in our course FAQ for **“queen gambit”**—it may be unrelated to the course content (or it’s a term we haven’t covered in the FAQ).

Can you tell me what course/topic this is for (e.g., chess, a lecture topic, or an assignment term)? If you meant something chess-related, we may need to confirm whether that’s in-scope for this course.  

Are there other areas you want to explore?


In [75]:
!uv add toyaikit

Resolved 129 packages in 2.89s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/74.86 KiB           
⠙ Preparing packages... (0/7)------------------- 14.81 KiB/74.86 KiB         
⠙ Preparing packages... (0/7)------------------- 14.81 KiB/74.86 KiB         
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/78.45 KiB           
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.45 KiB         
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠙ Preparing packages... (0/7)------------------- 14.84 KiB/78.45 KiB         
httpx2               ------------------------------ 14.81 KiB/74.86 KiB
⠙ Preparing p

In [76]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [77]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [78]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [79]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [80]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [82]:
chat_interface = IPythonChatInterface()

In [84]:

callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-nano")
)

In [85]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [86]:
result.cost

CostInfo(input_cost=Decimal('0.0008556'), output_cost=Decimal('0.00040375'), total_cost=Decimal('0.00125935'))

In [87]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run"}', call_id='call_amzkwrjnxzhfbtzHrX49

In [88]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run()